In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.cluster import DBSCAN

In [2]:
df = pd.read_csv("Market_Basket_Optimisation.csv", header=None)

print(df.shape)
df.head()

(7501, 20)


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,low fat yogurt,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
transactions = []

for i in range(len(df)):
    items = df.iloc[i].dropna().astype(str).tolist()
    transactions.append(items)

print(transactions[:5])

[['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes', 'whole weat flour', 'yams', 'cottage cheese', 'energy drink', 'tomato juice', 'low fat yogurt', 'green tea', 'honey', 'salad', 'mineral water', 'salmon', 'antioxydant juice', 'frozen smoothie', 'spinach', 'olive oil'], ['burgers', 'meatballs', 'eggs'], ['chutney'], ['turkey', 'avocado'], ['mineral water', 'milk', 'energy bar', 'whole wheat rice', 'green tea']]


In [4]:
mlb = MultiLabelBinarizer()

X = mlb.fit_transform(transactions)

encoded_df = pd.DataFrame(
    X,
    columns=mlb.classes_
)

print(encoded_df.shape)
encoded_df.head()

(7501, 120)


,asparagus,almonds,antioxydant juice,asparagus,avocado,babies food,bacon,barbecue sauce,black tea,blueberries,...,turkey,vegetables mix,water spray,white wine,whole weat flour,whole wheat pasta,whole wheat rice,yams,yogurt cake,zucchini
0,0,1,1,0,1,0,0,0,0,0,...,0,1,0,0,1,0,0,1,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,0


In [5]:
X_bool = X.astype(bool)

print(X_bool.shape)

(7501, 120)


In [6]:
dbscan = DBSCAN(
    eps=0.5,
    min_samples=5,
    metric="jaccard"
)

clusters = dbscan.fit_predict(X_bool)

print(clusters[:20])

[-1  0  0  0  0  0  0 -1  0  0  0  0  0  0  0  0 -1  0  0 -1]


In [7]:
result_df = df.copy()

result_df["Cluster"] = clusters

result_df.head(10)

,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,Cluster
0,shrimp,almonds,avocado,vegetables mix,green grapes,whole weat flour,yams,cottage cheese,energy drink,tomato juice,...,green tea,honey,salad,mineral water,salmon,antioxydant juice,frozen smoothie,spinach,olive oil,-1
1,burgers,meatballs,eggs,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,chutney,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,turkey,avocado,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,mineral water,milk,energy bar,whole wheat rice,green tea,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
5,low fat yogurt,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
6,whole wheat pasta,french fries,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
7,soup,light cream,shallot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1
8,frozen vegetables,spaghetti,green tea,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
9,french fries,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [8]:
unique_clusters = set(clusters)

number_of_clusters = len(unique_clusters) - (1 if -1 in unique_clusters else 0)

print("Number of clusters:", number_of_clusters)

Number of clusters: 1


In [9]:
cluster_counts = pd.Series(clusters).value_counts().sort_index()

print(cluster_counts)

-1    1252
 0    6249
Name: count, dtype: int64


In [10]:
noise_count = np.sum(clusters == -1)

print("Number of Noise Points:", noise_count)

Number of Noise Points: 1252


In [11]:
noise_percentage = (noise_count / len(df)) * 100

print("Noise Percentage:", round(noise_percentage, 2), "%")

Noise Percentage: 16.69 %


In [12]:
encoded_df["Cluster"] = clusters

cluster_labels = sorted(set(clusters))

for cluster in cluster_labels:

    if cluster == -1:
        continue

    cluster_data = encoded_df[
        encoded_df["Cluster"] == cluster
    ].drop("Cluster", axis=1)

    top_products = (
        cluster_data.mean()
        .sort_values(ascending=False)
        .head(10)
    )

    print("\nCluster", cluster)
    print("Top Products:")

    for product in top_products.index:
        print(product)


Cluster 0
Top Products:
mineral water
eggs
french fries
chocolate
spaghetti
green tea
milk
cookies
frozen vegetables
ground beef


In [13]:
result_df.to_csv(
    "DBSCAN_Clustered_Market_Basket.csv",
    index=False
)

print("DBSCAN clustered dataset saved successfully.")

DBSCAN clustered dataset saved successfully.
